# Model - save and export

# Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch

## Export - weights and model

In [ ]:
checkpoint_path = "../train/checkpoints/minifcos_face_v1_best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

state_dict = checkpoint["model_state_dict"]

export_dir = Path("export")
export_dir.mkdir(exist_ok=True)

weights_path = export_dir / "minifcos_face_v1_weights.npz"

weights = {
    name: tensor.detach().cpu().numpy()
    for name, tensor in state_dict.items()
}

np.savez_compressed(weights_path, **weights)

model_config = {
    "model_name": "MiniFCOS-Face-v1",
    "input_shape": [1, 3, 320, 320],
    "output_shape": [1, 6, 40, 40],
    "image_size": 320,
    "feature_size": 40,
    "stride": 8,
    "center_sampling_ratio": 0.5,
    "bbox_encoding": "ltrb_normalized_by_image_size",
    "channels": [
        "face_logit",
        "left",
        "top",
        "right",
        "bottom",
        "centerness_logit"
    ]
}

config_path = export_dir / "minifcos_face_v1_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(model_config, file, indent=4)

print("Težine:", weights_path)
print(f"Veličina: {weights_path.stat().st_size / 1024:.2f} KB")

print("\nKonfiguracija:", config_path)

print("\nPrvih 10 parametara:")
for index, (name, value) in enumerate(weights.items()):
    print(name, value.shape, value.dtype)

    if index == 9:
        break

## Export - layer references (debug purposes)

In [ ]:
from Model import MiniFCOSFaceV1 

checkpoint_path = "../train/checkpoints/minifcos_face_v1_best.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint["model_state_dict"]

model = MiniFCOSFaceV1()
model.load_state_dict(state_dict)
model = model.cpu()
model.eval()

with torch.no_grad():
    stem_pytorch = model.stem(x)
    block1_pytorch = model.block1(stem_pytorch)
    block2_pytorch = model.block2(block1_pytorch)
    head_features_pytorch = model.head[0](block2_pytorch)
    final_pytorch = model.head[1](head_features_pytorch)

reference_path = Path("export") / "pytorch_all_layer_references.npz"

np.savez_compressed(
    reference_path,
    input=x.cpu().numpy().astype(np.float32),
    stem_output=stem_pytorch.cpu().numpy().astype(np.float32),
    block1_output=block1_pytorch.cpu().numpy().astype(np.float32),
    block2_output=block2_pytorch.cpu().numpy().astype(np.float32),
    head_features_output=head_features_pytorch.cpu().numpy().astype(np.float32),
    final_output=final_pytorch.cpu().numpy().astype(np.float32)
)

print("Spremljeno:", reference_path)

print("Input         :", tuple(x.shape))
print("Stem          :", tuple(stem_pytorch.shape))
print("Block1        :", tuple(block1_pytorch.shape))
print("Block2        :", tuple(block2_pytorch.shape))
print("Head features :", tuple(head_features_pytorch.shape))
print("Final output  :", tuple(final_pytorch.shape))